In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import pandas as pd
import warnings
from IPython.display import display
current_dir = os.getcwd()
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project root added to Python path: {project_root}")

Project root added to Python path: c:\MariaSamosudova\Projects\UNIVER\repos\ELIQSIR\ipai_project


# **Check DWH connection**

In [2]:
load_dotenv(find_dotenv())

PROJECT_ROOT = os.getenv("PROJECT_ROOT")
IPAI_PROJECT_DIR = os.getenv("IPAI_PROJECT_DIR") # <-- Читаем новую директорию

if not IPAI_PROJECT_DIR:
    raise ValueError("IPAI_PROJECT_DIR is not set in the .env file!")

if IPAI_PROJECT_DIR not in sys.path:
    sys.path.append(IPAI_PROJECT_DIR)

from src.database.connection_manager import db_manager

print("Testing connection to AWS RDS...")

try:
    with db_manager.get_dwh_connection() as conn:
        with conn.cursor() as cursor:
            
            cursor.execute("SELECT VERSION();")
            version = cursor.fetchone()
            
            print(f"Success! Connected to AWS RDS.")
            print(f"Database version: {version[0]}")
            
            cursor.execute("SHOW DATABASES;")
            databases = [db[0] for db in cursor.fetchall()]
            print(f"Available databases: {databases}")
            
except Exception as e:
    print(f"Test error: {e}")

Testing connection to AWS RDS...
Success! Connected to AWS RDS.
Database version: 8.4.8
Available databases: ['eliqsir_dwh', 'information_schema', 'mysql', 'performance_schema', 'sys']


# **Build drug graphs**

In [2]:
from src.database.connection_manager import ConnectionManager
from src.retrieval.micro_layer.drugs.engine import QuantumGraphEngine
from src.retrieval.micro_layer.hdf5_handler import HDF5Handler
from src.retrieval.micro_layer.drugs.pipeline import DrugPipeline

db_manager = ConnectionManager()

engine = QuantumGraphEngine()
main_storage = HDF5Handler(filename="advanced_quantum_dataset.h5") 
pipeline = DrugPipeline(db_manager=db_manager, engine=engine, storage=main_storage)

print("Launching FULL dataset generation...")
print("Note: This process will take a significant(!!!) amount of time. You can monitor the progress bar.")
pipeline.run(batch_size=1000)

QuantumGraphEngine initialized: Strict 10-feature mode active.
HDF5 Storage initialized. Target file: c:\MariaSamosudova\Projects\UNIVER\repos\ELIQSIR\data\datasets\advanced_quantum_dataset.h5
Launching FULL dataset generation...
Note: This process will take a significant(!!!) amount of time. You can monitor the progress bar.
Starting Drug Graph Generation Pipeline...


Batch (offset 0):  33%|███▎      | 329/1000 [00:49<00:59, 11.23it/s][22:55:20] UFFTYPER: Unrecognized charge state for atom: 5
[22:55:20] UFFTYPER: Unrecognized charge state for atom: 5
Batch (offset 0):  39%|███▉      | 394/1000 [00:54<00:33, 18.35it/s][22:55:25] UFFTYPER: Unrecognized charge state for atom: 1
[22:55:25] UFFTYPER: Unrecognized charge state for atom: 1
Batch (offset 0):  51%|█████     | 512/1000 [01:10<03:12,  2.53it/s][22:55:41] UFFTYPER: Unrecognized charge state for atom: 6
[22:55:41] UFFTYPER: Unrecognized charge state for atom: 6
[22:55:41] UFFTYPER: Unrecognized charge state for atom: 6
[22:55:41] UFFTYPER: Unrecognized charge state for atom: 6
Batch (offset 0):  52%|█████▏    | 520/1000 [01:10<00:48,  9.88it/s][22:55:41] UFFTYPER: Unrecognized charge state for atom: 6
[22:55:41] UFFTYPER: Unrecognized charge state for atom: 6
Batch (offset 0):  52%|█████▏    | 523/1000 [01:10<00:36, 12.95it/s][22:55:41] UFFTYPER: Unrecognized charge state for atom: 6
[22:55:41] 


Critical Error saving batch to HDF5: [Errno 0] Unable to synchronously open file (unable to lock file, errno = 0, error message = 'No error', Win32 GetLastError() = 33)

PIPELINE COMPLETE!
Time elapsed: 46966.94 seconds
Successfully saved: 248413 graphs
Skipped (invalid SMILES or failed 3D/charges): 2591


In [ ]:
from src.database.connection_manager import ConnectionManager
from src.retrieval.micro_layer.drugs.engine import QuantumGraphEngine
from src.retrieval.micro_layer.hdf5_handler import HDF5Handler
from src.retrieval.micro_layer.drugs.pipeline import DrugPipeline

db_manager = ConnectionManager()

# Initialize Engine & Storage
# Important: HDF5Handler will automatically open the existing file in append ('a') mode, 
# so your 248k molecules are completely safe.
engine = QuantumGraphEngine()
main_storage = HDF5Handler(filename="advanced_quantum_dataset.h5") 

# Initialize Pipeline
pipeline = DrugPipeline(db_manager=db_manager, engine=engine, storage=main_storage)

# The exact offset where the crash happened, based on the logs
resume_offset = 251000

print(f"Resuming FULL dataset generation from offset {resume_offset}...")

# Run the pipeline starting from the resume_offset
# Note: Ensure your pipeline.run() method accepts the 'offset' parameter 
# and passes it to the SQL query (e.g., LIMIT batch_size OFFSET offset)
pipeline.run(batch_size=1000, start_offset=251000)

QuantumGraphEngine initialized: Strict 10-feature mode active.
HDF5 Storage initialized. Target file: c:\MariaSamosudova\Projects\UNIVER\repos\ELIQSIR\data\datasets\advanced_quantum_dataset.h5
Resuming FULL dataset generation from offset 251000...
Starting Drug Graph Generation Pipeline from offset 251000...


Batch (offset 251000):  48%|████▊     | 479/1000 [01:08<00:51, 10.19it/s][12:12:09] UFFTYPER: Unrecognized charge state for atom: 17
[12:12:09] UFFTYPER: Unrecognized charge state for atom: 17
Batch (offset 251000):  80%|███████▉  | 799/1000 [01:44<00:21,  9.36it/s][12:12:45] UFFTYPER: Unrecognized charge state for atom: 19
[12:12:45] UFFTYPER: Unrecognized charge state for atom: 19
Batch (offset 252000):  30%|██▉       | 299/1000 [01:30<02:02,  5.74it/s][12:14:45] UFFTYPER: Unrecognized charge state for atom: 19
[12:14:45] UFFTYPER: Unrecognized charge state for atom: 19
Batch (offset 253000):  38%|███▊      | 384/1000 [00:46<01:32,  6.69it/s][12:17:09] UFFTYPER: Unrecognized charge state for atom: 35
[12:17:09] UFFTYPER: Unrecognized charge state for atom: 35
Batch (offset 253000):  64%|██████▍   | 638/1000 [01:25<00:37,  9.74it/s][12:17:48] UFFTYPER: Unrecognized charge state for atom: 1
[12:17:48] UFFTYPER: Unrecognized charge state for atom: 1
Batch (offset 254000):  27%|██▋      